# WinCLIP Walkthrough

End-to-end walkthrough on a single MVTec-AD category. Run this top-to-bottom on a GPU machine with `WINCLIP_DATA_ROOT` set.

In [ ]:
import os, sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import yaml
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.datasets import MVTecAD
from src.methods import WinCLIP, build_prompt_ensemble
from src.metrics import image_auroc, pixel_auroc, per_region_overlap

In [ ]:
with open(REPO_ROOT / 'configs' / 'default.yaml') as f:
    cfg = yaml.safe_load(f)

category = 'bottle'
device = 'cuda'
shot = 0

data_root = Path(os.environ['WINCLIP_DATA_ROOT']) / 'mvtec_ad'
test_ds = MVTecAD(str(data_root), category, split='test')
print(f'Test samples: {len(test_ds)}')

## Inspect the prompt ensemble

In [ ]:
prompts = build_prompt_ensemble(category)
print(f"Normal prompts: {len(prompts['normal'])}")
print(f"Anomaly prompts: {len(prompts['anomaly'])}")
print()
print('Sample normal prompts:')
for p in prompts['normal'][:5]:
    print(f'  - {p}')
print()
print('Sample anomaly prompts:')
for p in prompts['anomaly'][:5]:
    print(f'  - {p}')

## Visualize a few test samples

In [ ]:
def denorm(t):
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3,1,1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3,1,1)
    return (t * std + mean).clip(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, idx in enumerate(np.random.choice(len(test_ds), 8, replace=False)):
    s = test_ds[int(idx)]
    axes[i//4, i%4].imshow(denorm(s['image']).permute(1,2,0))
    axes[i//4, i%4].set_title(s['defect'])
    axes[i//4, i%4].axis('off')
plt.tight_layout()

## Initialize WinCLIP and set up text anchors

In [ ]:
winclip = WinCLIP(
    model_name=cfg['backbone']['name'],
    pretrained=cfg['backbone']['pretrained'],
    scales=tuple(cfg['winclip']['scales']),
    device=device,
)
winclip.setup_text(category.replace('_', ' '))
print('Text anchors computed.')

## Score test samples and visualize heatmaps

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, idx in enumerate(np.random.choice(len(test_ds), 4, replace=False)):
    s = test_ds[int(idx)]
    img_score, score_map = winclip.score_image(
        s['image'].unsqueeze(0), target_size=240,
    )
    heat = score_map.squeeze().cpu().numpy()
    axes[0,i].imshow(denorm(s['image']).permute(1,2,0))
    axes[0,i].set_title(f"{s['defect']} (score={img_score.item():.3f})")
    axes[0,i].axis('off')
    axes[1,i].imshow(heat, cmap='jet')
    axes[1,i].axis('off')
plt.tight_layout()

## Evaluate on the full test set

In [ ]:
loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)
image_scores, labels, score_maps, gt_masks = [], [], [], []
for batch in loader:
    img_score, score_map = winclip.score_image(batch['image'], target_size=240)
    image_scores.extend(img_score.cpu().numpy().tolist())
    labels.extend(batch['label'].cpu().numpy().tolist())
    score_maps.extend(score_map.cpu().numpy())
    gt_masks.extend(batch['mask'].cpu().numpy())

print(f'Image-AUROC: {image_auroc(np.array(image_scores), np.array(labels))*100:.2f}')
print(f'Pixel-AUROC: {pixel_auroc(score_maps, gt_masks)*100:.2f}')
print(f'Pixel-AUPRO: {per_region_overlap(score_maps, gt_masks)*100:.2f}')